In [32]:
import pandas as pd

Data loading

In [33]:
TRAIN_PATH = "/Users/david/Documents/data/jusbrasil/manual/with_body/splits/train.parquet"
VAL_PATH = "/Users/david/Documents/data/jusbrasil/manual/with_body/splits/vali.parquet"
TEST_PATH = "/Users/david/Documents/data/jusbrasil/manual/with_body/splits/small_test.parquet"

In [34]:
train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(VAL_PATH)
test_df = pd.read_parquet(TEST_PATH)

Prepocessing

In [35]:
FEATURES_COLUMNS_PATH = "/Users/david/Documents/data/jusbrasil/feature_map.json"
FEATURES_COLUMNS = pd.read_json(FEATURES_COLUMNS_PATH)[0].tolist()

In [36]:
def fill_na_features(df, columns):
    df[columns] = df[columns].fillna(0) 
    return df

train_df = fill_na_features(train_df, FEATURES_COLUMNS)
test_df = fill_na_features(test_df, FEATURES_COLUMNS)
val_df = fill_na_features(val_df, FEATURES_COLUMNS)

In [37]:
def preprocess_lambdamart(df, features_columns=FEATURES_COLUMNS):
    df['query_id'] = pd.factorize(df['query'])[0]
    
    X = df[features_columns]
    y = df['relevance']
    group = df.groupby('query_id', sort=False).size().to_numpy()
    return X, y, group

In [ ]:
X_train, y_train, group_train = preprocess_lambdamart(train_df)
X_test, y_test, group_test = preprocess_lambdamart(test_df)
X_val, y_val, group_val = preprocess_lambdamart(val_df)

Training LambdaMART

In [39]:
import lightgbm as lgb

lgb_train = lgb.Dataset(X_train, y_train, group=group_train)
lgb_val = lgb.Dataset(X_val, y_val, group=group_val)

In [40]:
params = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'ndcg_eval_at': 10,
    'learning_rate': 0.01,
    'num_leaves': 31,
    'verbose': -1,
}

model = lgb.train(params, 
lgb_train, 
valid_sets=[lgb_val], 
num_boost_round=100, 
callbacks=[lgb.early_stopping(stopping_rounds=10)])

Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[32]	valid_0's ndcg@10: 0.855008


In [41]:
test_df['score'] = model.predict(X_test)

Evaluating results

In [42]:
import numpy as np

def ndcg_at_k(labels_sorted_by_score: np.ndarray, labels_ideal: np.ndarray, k: int) -> float:
    k = min(k, len(labels_sorted_by_score))
    predicted = labels_sorted_by_score[:k]
    ideal = np.sort(labels_ideal)[::-1][:k]

    discounts = np.log2(np.arange(1, k + 1) + 1)
    dcg = np.sum((np.power(2.0, predicted) - 1) / discounts)
    idcg = np.sum((np.power(2.0, ideal) - 1) / discounts)
    return 0.0 if idcg == 0.0 else dcg / idcg


ks = [1, 3, 5, 10]
rows = []

for qid, g in test_df.groupby("query_id"):
    y_q = g["relevance"].to_numpy(dtype=float)
    scores = g["score"].to_numpy(dtype=float)

    y_ranked = y_q[np.argsort(scores)[::-1]]

    row = {"query_id": qid, "num_docs": len(g)}
    for k in ks:
        row[f"ndcg@{k}"] = ndcg_at_k(y_ranked, y_q, k)
    rows.append(row)

metrics = pd.DataFrame(rows)
for k in ks:
    print(f"NDCG@{k:<3} = {metrics[f'ndcg@{k}'].mean():.4f}  (over {len(metrics)} queries)")

NDCG@1   = 0.8667  (over 30 queries)
NDCG@3   = 0.8784  (over 30 queries)
NDCG@5   = 0.8679  (over 30 queries)
NDCG@10  = 0.8814  (over 30 queries)
